# Modelado de escenarios e impacto en negocio 🧮💡


## Objetivos académicos

---




* Comprender cómo los funnels de conversión permiten analizar el comportamiento de los usuarios a lo largo del proceso de compra.
* Aprender a **simular mejoras** en distintas etapas del funnel usando SQL y evaluar su impacto total.
* Estimar el **impacto económico** de una mejora a partir de métricas como ARPU o revenue.
* Comparar resultados entre **segmentos o fuentes de tráfico** para priorizar oportunidades de optimización.
* Desarrollar una mentalidad analítica orientada a la toma de decisiones basada en datos y escenarios hipotéticos.

> 🎯 *Meta general:* que los estudiantes puedan modelar escenarios tipo “¿qué pasaría si?” usando SQL y traducirlos en métricas de negocio comprensibles para stakeholders.


## ¿Qué es una simulación de escenarios en un funnel? 🧠

---


Una simulación de escenarios es una forma de **probar “qué pasaría si” algo cambiara dentro del embudo de conversión**, sin necesidad de modificar los datos reales.  
Imagina que tu sitio web tiene muchas visitas, pero pocos usuarios llegan a comprar. Con una simulación, puedes estimar **qué ocurriría si mejoraras una etapa del proceso**, por ejemplo, reduciendo el abandono al ver productos o aumentando la tasa de compra.  

En otras palabras, es como crear un *laboratorio virtual del negocio* 🔬: usamos datos históricos, aplicamos un cambio hipotético (como una mejora del 10% o una reducción del abandono) y observamos **cómo ese cambio se propaga por todo el funnel y en las métricas de negocio**.


### 🧩 Ejercicio práctico: simulando mejoras en el funnel con la base *journeys*

Vamos a trabajar con la tabla `events`, que contiene los eventos del recorrido de los usuarios:  
`session_start → view_item → add_to_cart → begin_checkout → purchase`

Este conjunto de datos nos permitirá **simular una mejora** en el comportamiento de los usuarios y observar su impacto a lo largo del funnel.

---


#### 🎯 Objetivo
Simular qué pasaría si reducimos en **20% el abandono entre las etapas `view_item` y `add_to_cart`**.  
Queremos ver cómo esa mejora se propaga hasta las compras finales.

---

#### 🧠 Lógica del ejercicio

1. Calculamos la línea base (baseline) de usuarios en cada etapa.
2. Aplicamos la mejora del 20% sobre la pérdida entre `view_item` y `add_to_cart`.
3. Propagamos los cambios a las siguientes etapas, manteniendo las tasas de conversión históricas.
4. Mostramos los resultados comparando el funnel original con el simulado.

---

#### 💻 Query de simulación

```sql
WITH data AS (
  SELECT 
    event_name,
    CASE
      WHEN event_name = 'session_start' THEN 1
      WHEN event_name = 'view_item' THEN 2
      WHEN event_name = 'add_to_cart' THEN 3
      WHEN event_name = 'begin_checkout' THEN 4
      ELSE 5
    END AS order_event,
    COUNT(DISTINCT user_id) AS users
  FROM events
  GROUP BY event_name
),
baseline AS (
  SELECT 
    MAX(CASE WHEN event_name = 'session_start' THEN users END) AS session_start,
    MAX(CASE WHEN event_name = 'view_item' THEN users END) AS view_item,
    MAX(CASE WHEN event_name = 'add_to_cart' THEN users END) AS add_to_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN users END) AS begin_checkout,
    MAX(CASE WHEN event_name = 'purchase' THEN users END) AS purchase
  FROM data
)
-- 🚀 Simulación: reducimos 20% del abandono entre view_item → add_to_cart
SELECT
  session_start AS session_start_base,
  view_item AS view_item_base,
  add_to_cart AS add_to_cart_base,
  begin_checkout AS begin_checkout_base,
  purchase AS purchase_base,

  -- 1️⃣ Aplicamos la mejora (solo queda el 80% del abandono original)
  ROUND(view_item - ((view_item - add_to_cart) * 0.8)) AS add_to_cart_simulated,

  -- 2️⃣ Propagamos la mejora manteniendo las tasas históricas
  ROUND(
    (view_item - ((view_item - add_to_cart) * 0.8)) 
    * (begin_checkout * 1.0 / add_to_cart)
  ) AS begin_checkout_simulated,

  ROUND(
    (view_item - ((view_item - add_to_cart) * 0.8))
    * (begin_checkout * 1.0 / add_to_cart)
    * (purchase * 1.0 / begin_checkout)
  ) AS purchase_simulated
FROM baseline;


#### Sigamos practicando 🤔

---

1. Supón que ahora quieres simular una mejora del *+15%* en la etapa `begin_checkout → purchase`. ¿Cómo cambiarías el query para reflejar este nuevo escenario? ¿Qué columna modificarías?

1. ¿Por qué es importante mantener las tasas históricas al propagar una mejora en la simulación?

1. ¿Cuál es la diferencia entre *tasa de conversión* y *tasa de abandono* dentro de un funnel? 

1. 2️⃣ **Reflexión**: Si al simular mejoras obtienes un incremento pequeño en compras totales, ¿qué te dice eso sobre la etapa en la que aplicaste la mejora y su influencia dentro del funnel?

## Estimando el impacto en métricas del negocio 💰📈

---



Cuando simulamos mejoras en un funnel, no solo buscamos entender cuántos usuarios adicionales completan una acción, sino **cuánto valor económico generan esos cambios**.  
Esto se logra estimando el **impacto en métricas de negocio** como ingresos, ARPU (*Average Revenue Per User*), o número adicional de compras.  

Para hacerlo:
1. Partimos de la simulación anterior y calculamos cuántos usuarios más completan la compra.  
2. Multiplicamos ese incremento por el **ARPU**, que representa el ingreso promedio por usuario.  
3. Así obtenemos una estimación del **impacto económico total**, sin haber hecho aún ningún cambio real en el producto.  

Esta práctica es esencial porque permite **priorizar las mejoras con mayor retorno económico** y comunicar el valor de una optimización en términos comprensibles para los equipos de negocio o marketing. 🚀  

---

### 💻 Ejemplo práctico: impacto económico de una mejora en *journeys*

Supongamos que queremos estimar el impacto de una **mejora del +10% en la tasa de conversión entre `begin_checkout` y `purchase`**, y que el **ARPU = 200 USD**.

Queremos saber cuántas compras adicionales obtendríamos y cuánto ingreso representarían.

```sql
WITH data AS (
  SELECT 
    event_name,
    COUNT(DISTINCT user_id) AS users
  FROM events
  GROUP BY event_name
),
baseline AS (
  SELECT
    MAX(CASE WHEN event_name = 'session_start' THEN users END) AS session_start,
    MAX(CASE WHEN event_name = 'view_item' THEN users END) AS view_item,
    MAX(CASE WHEN event_name = 'add_to_cart' THEN users END) AS add_to_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN users END) AS begin_checkout,
    MAX(CASE WHEN event_name = 'purchase' THEN users END) AS purchase
  FROM data
),
simulated AS (
  SELECT
    begin_checkout,
    purchase,
    -- 💡 Incrementamos en 10% la brecha entre begin_checkout y purchase
    purchase + ROUND((begin_checkout - purchase) * 0.10) AS purchase_simulated
  FROM baseline
)
SELECT
  begin_checkout,
  purchase AS purchase_base,
  purchase_simulated,
  (purchase_simulated - purchase) AS additional_purchases,
  (purchase_simulated - purchase) * 200 AS revenue_impact_usd
FROM simulated;


####  Sigamos practicando 🤔

---

1. Ahora supón que el ARPU sube a **250 USD**, pero la mejora solo es del **+5%**.  
¿Cómo ajustarías el cálculo para estimar el nuevo impacto económico?

1. ¿Qué es el ARPU y por qué es una métrica clave al evaluar el impacto de una simulación en el negocio?

1. 2️⃣ **Reflexión**: Si dos etapas del funnel muestran el mismo aumento en conversiones, pero una tiene un impacto económico mayor, ¿qué implicación tiene eso para la priorización de esfuerzos?

## Comparando escenarios entre segmentos 🎯📊

---


En un funnel, no todos los usuarios se comportan igual. Los resultados pueden variar dependiendo del **segmento**, como la fuente de tráfico, el dispositivo o la campaña publicitaria.  
Comparar escenarios entre segmentos nos permite identificar **dónde una mejora tendría mayor impacto** y así enfocar los recursos en los grupos más rentables o con mayor oportunidad de optimización.

Para hacerlo:
1. Agrupamos los datos por un campo de segmentación (por ejemplo, `stream_id` o `source`).  
2. Calculamos el funnel base para cada grupo.  
3. Simulamos una mejora y estimamos su impacto por separado.  
4. Comparamos los resultados para ver cuál segmento genera más valor o responde mejor a los cambios.  

Esta técnica es clave para **priorizar experimentos A/B**, ya que muestra en qué grupos una mejora inicial puede traducirse en un mayor retorno económico. 💡

---

### 💻 Ejemplo práctico: simulación por fuente de tráfico en *journeys*

Queremos analizar qué pasaría si cada fuente de tráfico (`stream_id`) mejora un **10% su conversión inicial** (de `session_start` a `view_item`), con un **ARPU = 150 USD**.  
El objetivo es identificar cuáles fuentes generarían más ingresos adicionales con esa mejora.

```sql
WITH base AS (
  SELECT
    stream_id,
    COUNT(DISTINCT CASE WHEN event_name = 'session_start' THEN user_id END) AS session_start,
    COUNT(DISTINCT CASE WHEN event_name = 'view_item' THEN user_id END) AS view_item,
    COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END) AS add_to_cart,
    COUNT(DISTINCT CASE WHEN event_name = 'begin_checkout' THEN user_id END) AS begin_checkout,
    COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END) AS purchase
  FROM events
  GROUP BY stream_id
),
simulated AS (
  SELECT
    stream_id,
    session_start,
    view_item + ROUND((session_start - view_item) * 0.10) AS view_item_simulated,
    add_to_cart + ROUND((session_start - view_item) * 0.10 * (add_to_cart * 1.0 / view_item)) AS add_to_cart_simulated,
    begin_checkout + ROUND((session_start - view_item) * 0.10 * (add_to_cart * 1.0 / view_item) * (begin_checkout * 1.0 / add_to_cart)) AS begin_checkout_simulated,
    purchase + ROUND((session_start - view_item) * 0.10 * (add_to_cart * 1.0 / view_item) * (begin_checkout * 1.0 / add_to_cart) * (purchase * 1.0 / begin_checkout)) AS purchase_simulated,
    purchase AS purchase_base
  FROM base
)
SELECT
  stream_id,
  (purchase_simulated - purchase_base) AS additional_purchases,
  (purchase_simulated - purchase_base) * 150 AS revenue_impact_usd
FROM simulated
ORDER BY revenue_impact_usd DESC;

```


####  Sigamos practicando 🤔

---



1. Ahora supón que la mejora del **10%** solo se aplica a las campañas móviles (`stream_id` con “mobile”).  
   ¿Cómo ajustarías el `WHERE` o la segmentación en el query para analizar únicamente ese grupo?

1. 💡 **Concepto:** ¿Por qué es importante segmentar los funnels antes de tomar decisiones sobre optimización o inversión publicitaria?

1. 2️⃣ **Reflexión:** Si un segmento tiene un impacto potencial bajo, pero representa un público estratégico (por ejemplo, nuevos usuarios), ¿cómo equilibrarías el valor económico frente al valor estratégico?


## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta  nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [`Discord`](https://discord.com/channels/1081207584104656986/1420849538196836472).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal [`#project`](https://discord.com/channels/1081207584104656986/1420848813186351134) para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
    - En tus preguntas recuerda etiquetar a `@Dataconsulta` y ubica tu pregunta de acuerdo a `Sprint/Capitulo/Seccion`
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨